In [2]:
!pip install -q google-genai faiss-cpu      #faiss- it is a liibrary desgined for searching and comparing vectors and cpu is a version of faiss

import faiss
import numpy as np   #numpy is mainly for working with numerical array, it makes it easy to store and perform mathematical operations on these numbers

from google import genai
from google.genai import types
from google.colab import userdata

client=genai.Client(api_key=userdata.get('Ragproject'))

EMBEDDING_MODEL='gemini-embedding-001'
EMBEDDING_DIMENSION=768  #this tells your program the size of the embedding vector you're going to work with


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 51.9 MB/s eta 0:00:00


In [3]:
#_____________EMBEDDING_FUNCTION_____________________#
def embed(text):
  response=client.models.embed_content(
      model=EMBEDDING_MODEL,
      contents=text,
      config=types.EmbedContentConfig(
          output_dimensionality=EMBEDDING_DIMENSION,
          )
      )
  return(response.embeddings[0].values) #[0] 1st embedding obj from the list , .value gets the 768 number belonging to that embedding

In [4]:
text="React developer with payment gateway experience"

embedding=embed(text)

print("Type: ",type(embedding))
print("Length: ",len(embedding))
print("Fist few numbers: ",embedding[:8])

Type:  <class 'list'>
Length:  768
Fist few numbers:  [-0.004400805, -0.01087601, 0.028694738, -0.08073173, -0.023548044, 0.014271031, 0.044820398, 0.0137446225]


In [5]:
sentence=[                 #list containing 3 string
    "React developer with paymet gateway experience",
    "Built UPI checkout flow for Razerpay using Next.js and Node",
    "Cardiologist with 12 year experience at AIIMS Delhi"
]

embedding=np.array(
    [embed(sentence) for sentence in sentence]  #for each sentence insdie sentence[], we call embed(sentence)
) #internally it will be like embed(sentence[0]),embed(sentence[1]),embed(sentence[2]) and get embedding:1 -768 num, embedding:2 -768 num, embedding:3- 768 num , this is called as list comprehension

print("Embedding shape: ",embedding.shape) #wht is the size structre of the array

Embedding shape:  (3, 768)


In [12]:
# ______________COSINE_SIMILARITY___________________________#

def cosine_sim(a, b):
    a = a    #np.array(a)
    b = b    #np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


sim_react_nextjs = cosine_sim(embedding[0], embedding[1])
sim_react_doctor = cosine_sim(embedding[0], embedding[2])
sim_nextjs_doctor = cosine_sim(embedding[1], embedding[2])

print("========== COSINE SIMILARITY ==========")
print(f"React vs Next.js: {sim_react_nextjs}") #without decimal formatting
print(f"React vs Cardiologist: {sim_react_doctor:.4f}")
print(f"Next.js vs Cardiologist: {sim_nextjs_doctor:.4f}")

========== COSINE SIMILARITY ==========
React vs Next.js: 0.6212732692118814
React vs Cardiologist: 0.4801
Next.js vs Cardiologist: 0.4332


In [20]:
#__________________LIST OF CANDIDATES_________________________#

candidates = [
    "Built UPI checkout flow for Razorpay using Next.js and Node",
    "Cardiologist with 12 year experience at AIIMS Delhi",
    "Senior backend engineer, Kubernetes + Go at Zomato",
    "iOS developer, Swift, built food delivery app for Swiggy",
    "MBBS doctor, internal medicine, Manipal Hospital",
    "DevOps engineer, AWS ECS, Terraform, fintech background"
]

#converting candidates into embedding that gives 768 num
candidate_vector = np.array([embed(candidate) for candidate in candidates])
print("Candidate vector shape: ", candidate_vector.shape)

#Normalize - normaliz every candidate vector so that its length becomes one
candidate_vector = candidate_vector / np.linalg.norm( #taking each candidate vector and dividing by its own length
    candidate_vector,
    axis=1, #tells Numpy work across each row, here each row is one candidate vector    #axis=0 works with column
    keepdims=True  #keeps the output dimension/shape so it can be used for further operations like division
)
print("Normalized candidate vector: ", candidate_vector.shape)


#___________SEARCH____________
def search(query, top_k=3):
    q_vec = np.array(embed(query))   #embedding the query to 768 number
    q_vec = q_vec / np.linalg.norm(q_vec) #normalizing the query

    scores = candidate_vector @ q_vec      # @-means matrix multiplication

    #finding the highest score
    top_indices = np.argsort(scores)[::-1][:top_k]  #np.argsort - findes the position of the score in sorted order
                                                    #[::-1] - reverse that order, because we want highest to lowest
                                                    #[:top_k] - if top_k=3 then only first 3
                                                    #so, scores -> sort -> hightest first -> take top 3
    return [

        {
            "candidate": candidates[index],
            "score": float(scores[index])
        }
        for index in top_indices
    ]


#_______test query____________#
queries = [
    "React dev with payment experience",
    "Senior heart specialist",
    "Cloud infrastructure for food delivery"
]

for i, query in enumerate(queries, 1):
    print(f"\n_________FAISS Query {i}_______\n{query}")

    for result in search(query):
        print(f"{result['score']:.3f} - {result['candidate']}")

Candidate vector shape:  (6, 768)
Normalized candidate vector:  (6, 768)

_________FAISS Query 1_______
React dev with payment experience
0.653 - DevOps engineer, AWS ECS, Terraform, fintech background
0.599 - Built UPI checkout flow for Razorpay using Next.js and Node
0.559 - iOS developer, Swift, built food delivery app for Swiggy

_________FAISS Query 2_______
Senior heart specialist
0.699 - Cardiologist with 12 year experience at AIIMS Delhi
0.589 - MBBS doctor, internal medicine, Manipal Hospital
0.504 - DevOps engineer, AWS ECS, Terraform, fintech background

_________FAISS Query 3_______
Cloud infrastructure for food delivery
0.636 - iOS developer, Swift, built food delivery app for Swiggy
0.616 - Senior backend engineer, Kubernetes + Go at Zomato
0.566 - DevOps engineer, AWS ECS, Terraform, fintech background
